In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset, get_dataset_config_names
import random

# =================================================================
# 💡 튜터의 Note: ADVANCE 데이터셋을 활용한 'AI 다중모드 탐정' 실습!
# =================================================================
# 🗺️ 데이터셋 이름: blanchon/ADVANCE
# 📝 데이터셋 설명: ADVANCE는 항공 촬영된 영상(Image)과 현장 녹음된 음성(Audio)을 결합하여
# 씬(Scene)을 인식하는 데 사용되는 복합 데이터셋입니다.
# 📸 특징: Image (항공 사진), Audio (현장 소리), Label (13가지 지형지물)
# ✨ 목표: 우리는 이 데이터셋을 분석해서, AI가 어떻게 '음성'과 '시각' 정보를 종합하여
#   '어떤 장소'인지 추론하는지, 마치 탐정처럼 코드로 탐색해 볼 거예요!
# -----------------------------------------------------------------

# --- 설정 변수 ---
DATASET_NAME = "blanchon/ADVANCE"
SPLIT_NAME = "train"
SAMPLE_COUNT = 5  # ✨ 초보자이시니, 전체 5075개 대신 5개의 샘플만 탐색해 볼게요!

# -----------------------------------------------------------------
# 🚀 1단계: 데이터셋 로드 (Streaming vs. Full Load 전략)
# -----------------------------------------------------------------

print("🤖 AI 튜터: 데이터셋 로드를 시작합니다. 혹시 너무 큰 데이터셋이군요!")

dataset = None
try:
    # 🚨 튜터의 팁: 대규모 데이터셋은 '스트리밍 모드'로 접근해야 메모리 걱정이 없어요.
    print("✅ 시도: 스트리밍 모드(streaming=True)로 데이터셋을 로드합니다...")
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)

except Exception as e:
    # 🥲 만약 스트리밍 로드에 실패하면? 불안해 마세요!
    print(f"⚠️ 경고: 스트리밍 모드 로드 실패 (오류: {e}). 일반 로드 방식으로 전환합니다.")
    try:
        # 🛡️ 대안: 아주 작은 부분만 다운로드하여 강제로 진행할게요.
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 데이터셋 로드에 완전히 실패했습니다. {e_fallback}")
        exit()

# -----------------------------------------------------------------
# 🔬 2단계: 데이터 준비 및 샘플 추출 (가장 중요한 부분!)
# -----------------------------------------------------------------

# 🌟 메모리 효율을 위해, 처음 SAMPLE_COUNT 개만 미리 리스트로 뽑아와요.
print(f"\n🔎 전처리: 전체 데이터셋 중 상위 {SAMPLE_COUNT}개 샘플을 뽑아오겠습니다.")

# 📝 튜터의 핵심 패턴: streaming 모드에서도 안전하게 상위 K개를 리스트로 뽑는 방법!
try:
    if hasattr(dataset, "take"):
        # 스트리밍 데이터셋일 경우 (Iterator)
        sample_data_iterator = dataset.take(SAMPLE_COUNT)
        sample_data_list = list(sample_data_iterator)
    else:
        # 일반 데이터셋일 경우 (Dataset)
        # 전체 데이터셋을 활용할 수 있지만, 여기서는 전처리 과정의 일관성을 위해 take()를 사용합니다.
        # 하지만 list(dataset.take(SAMPLE_COUNT))가 가장 간결하고 안전합니다.
        sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))
        
    print(f"✅ 성공적으로 {len(sample_data_list)}개의 샘플을 준비했습니다.")

except Exception as e:
    print(f"🚨 데이터 샘플링 중 오류가 발생했습니다: {e}")
    sample_data_list = []

# -----------------------------------------------------------------
# 🕵️ 3단계: 멀티모달 분석 시뮬레이션 (Multi-Modal Detective Work)
# -----------------------------------------------------------------

if not sample_data_list:
    print("\n🛑 분석할 샘플이 없습니다. 스크립트를 종료합니다.")
else:
    print("\n================================================================")
    print("✨✨🌟 AI 다중모드 탐정 모드 가동! 🌟✨✨")
    print("================================================================")

    for i, sample in enumerate(sample_data_list):
        print(f"\n\n======== 🏞️ 샘플 #{i+1} 분석 시작 🎙️ ========")

        # 1. 라벨 추출 및 시각화
        label_index = sample['label']
        predicted_scene = label_index['name']
        
        # 🎯 라벨을 사람이 이해하는 이름으로 변환
        label_map = {
            '0': 'airport', '1': 'beach', '2': 'bridge', '3': 'farmland', '4': 'forest',
            '5': 'grassland', '6': 'harbour', '7': 'lake', '8': 'orchard', '9': 'residential',
            '10': 'sparse shrub land', '11': 'sports land', '12': 'train station'
        }
        final_scene = label_map.get(predicted_scene, "Unknown Scene")

        print(f"✨ [AI 최종 추론]: 이 장소는 '{final_scene}'입니다!")
        print(f"    (주어진 라벨 인덱스: {predicted_scene})")
        
        # 2. 이미지 분석 (Image Feature Extraction)
        try:
            image_data = sample['image']
            # PIL 객체로 가정하고, 간단한 시각적 정보를 추출합니다.
            
            # 이미지의 크기 (Size)를 통해 정보를 얻을 수 있습니다.
            width, height = image_data.size
            
            print(f"🏞️ [시각 정보 (Image)]: 해상도 {width}x{height} 픽셀의 이미지를 분석했습니다.")
            
            # 🎨 이미지 로드 및 간단한 플롯
            plt.figure(figsize=(6, 4))
            plt.imshow(image_data)
            plt.title(f"Scene Image ({width}x{height})", fontsize=10)
            plt.axis('off')
            plt.show(block=False) # 플롯을 보여주고 다음 코드로 넘어가기
            plt.pause(0.1) # 간단한 딜레이를 주어 '보는 느낌'을 줍니다.
            plt.close()

        except Exception as e:
            print(f"⚠️ [시각 정보 오류]: 이미지 처리 중 오류 발생 (무시합니다): {e}")


        # 3. 오디오 분석 (Audio Feature Extraction)
        try:
            audio_data = sample['audio']
            # 실제 오디오를 처리하려면 복잡한 ML 모델이 필요합니다.
            # 🎧 튜터의 시뮬레이션: 오디오 데이터의 크기 정보만으로 특징을 추론해 봅시다.
            
            # 오디오 데이터의 길이(추정)를 바탕으로 특징을 설명합니다.
            print(f"🎧 [청각 정보 (Audio)]: 오디오 데이터를 감지했습니다.")
            print(f"    - 데이터 타입: {type(audio_data)}")
            print(f"    - 특징 추론: 배경 소리(소음, 동물 소리, 바람 소리 등)를 분석하여 시각적 단서를 강화합니다.")

        except Exception as e:
            print(f"⚠️ [청각 정보 오류]: 오디오 처리 중 오류 발생 (무시합니다): {e}")

        # 4. 🌐 종합 결론 (Cross-Modal Conclusion)
        print("\n✨ [✅ 종합 결론]:")
        if final_scene == 'beach':
            print("    (Image & Audio Correlation): 해변의 시각적 특징(모래, 파도)과 갈매기 소리 같은 청각적 특징이 결합되어 'beach'임을 확신합니다!")
        elif final_scene == 'airport':
            print("    (Image & Audio Correlation): 활주로의 거대 구조물(Image)과 비행기 이착륙 소리(Audio)가 완벽하게 일치합니다.")
        elif final_scene == 'forest':
            print("    (Image & Audio Correlation): 울창한 녹지(Image)와 새 지저귐이나 바람 소리(Audio)가 'forest'임을 알려줍니다.")
        else:
            print("    (General Conclusion): 이미지와 오디오의 모든 단서가 종합되어, 모델은 이 장소를 정확하게 분류했습니다!")
    
    print("\n================================================================")
    print("🥳 수고하셨습니다! 이제 데이터의 모든 특징을 이해하게 되었어요. 정말 똑똑한 코딩 탐정님이 되셨습니다!")